In [ ]:
#| hide
from dataset_archiver.core import *

# Dataset Archiver for WebDAV

Uploads a research dataset folder to the UniPi u-drive (a Nextcloud WebDAV server) as a dated, content-hashed archive
folder, together with a metadata file, a file manifest, a licence and a README, and records the upload in a shared
registry. Uploads can be interrupted and resumed, which matters for datasets of tens of thousands of files.

## If you are coming back to this after a while

Everything is done with one command, `upload-dataset`, run from the root of this repository (the folder that contains
`upload_registry.csv`). Pick the case that matches what you want to do:

| I want to... | Go to |
|---|---|
| archive a new dataset | [Archiving a new dataset](#archiving-a-new-dataset) |
| finish an upload that stopped or that I killed | [Resuming an upload](#resuming-an-upload) |
| know how far an upload got, without changing anything | [Checking where an upload stands](#checking-where-an-upload-stands) |
| understand an error or a run that seems stuck | [Troubleshooting](#troubleshooting) |
| change the code | [Editing the code](#editing-the-code) |

Check first that the command is installed and up to date. The output should list `--archive-date` and `--version-hash`:

```bash
upload-dataset --help | grep -E "archive-date|version-hash"
```

If the command is missing, run `pip install -e .` from this folder, in the Python environment you use for this project.

## Why this exists

Some research datasets have to be archived so that they cannot change afterwards, but cannot go to an open repository
such as Zenodo or OSF because they are too large, restricted by an agreement, or not public. This tool stores such a
dataset on the university's u-drive in a folder whose name includes the upload date and a hash of the contents, keeps the
data and its description together, attaches a restricted-use licence, and records every archive in one shared registry.

## Archiving a new dataset

1. Run a dry run first. It logs in, lists what is already on the server and prints the plan, but sends no uploads:

```bash
cd ~/Downloads/CLAUDE/dataset-archiver
upload-dataset \
  --source "/full/path/to/DATASET" \
  --label 2026_DATASETNAME \
  --title "Short title" \
  --source_name "Who collected the data" \
  --date_collected "2026-05-01--2026-05-03" \
  --description "What the data is, where and how it was collected, anything a future user must know." \
  --responsible "Andrea Munafo" \
  --username a012197 \
  --email andrea.munafo@unipi.it \
  --webdav https://u-drive.unipi.it/remote.php/webdav \
  --dry-run
```

2. Run the same command without `--dry-run` and with `--verify`. On a Mac, prefix it with `caffeinate -is` so the
machine does not sleep during a long upload, and pipe it through `tee` to keep a log:

```bash
caffeinate -is upload-dataset ...same arguments... --verify 2>&1 | tee "upload_DATASETNAME_$(date +%Y%m%d_%H%M).log"
```

3. Write down the two values printed near the top of the output. You need them if the upload stops:

```
📁 Remote folder: https://u-drive.unipi.it/.../archival_data/2026_DATASETNAME_20260927_1802d4f6
   To resume into this folder later: --archive-date 20260927 --version-hash 1802d4f6
```

The password prompt asks for your u-drive password. If the account uses university single sign-on or two-factor
login, create an app password in the u-drive web interface (Settings, Security) and use that instead.

The title, description and other metadata go into `metadata.json` and `README.txt` on the server and into the registry,
so write the description for someone who has never seen the data.

A run is finished when it prints `✅ Upload complete.`, the verification shows no missing files and no size
differences, and the command exits with status 0.

## Resuming an upload

An upload can stop because of the network, because the server is rate limiting, because the laptop slept, or because you
pressed Ctrl-C. Nothing is lost: every file that reached the server stays there, and the next run skips it.

To resume, run exactly the same command again and add the two pinning options:

```bash
upload-dataset ...same arguments as the first run... --archive-date 20260927 --version-hash 1802d4f6 --dry-run
upload-dataset ...same arguments as the first run... --archive-date 20260927 --version-hash 1802d4f6 --verify
```

The dry run is optional but cheap: it shows how many files are already on the server and how many are left.

Where to find the date and hash if you did not write them down:

- the first lines of the log of the earlier run (`To resume into this folder later: ...`);
- the folder name on u-drive, under `Robotics-Research-Lab/archival_data/`: it is `LABEL_YYYYMMDD_HASH`;
- the name of a failure log in the repository folder, `upload_failures_LABEL_YYYYMMDD_HASH_<time>.tsv`.

Why the pins matter. Without `--archive-date`, the folder name uses today's date, so a run on a different day would start
a second, separate archive folder. Without `--version-hash`, the tool re-reads every byte of the dataset to compute the
hash (about half an hour for 100 GB), and the result can differ from the original if macOS has since created hidden
files such as `.DS_Store` in the dataset (see [Known issues](#known-issues)). With `--version-hash` given, the tool
checks that the folder already exists on the server and stops if it does not, so a typo cannot create a new folder.

Use the same `--label` and the same metadata as the first run. The label is part of the folder name, and the metadata
is written into the sidecar files at the end.

If a resumed run reports files `on server with a DIFFERENT size`, it leaves them alone and does not mark the upload as
complete. Look at those files (they are listed in the failure log). If the local copy is the right one, run once more with
`--overwrite-mismatched` to replace them.

## Checking where an upload stands

Run the resume command with `--dry-run`. It only reads from the server and prints a plan such as:

```
📋 Plan: 4616 already on server (same size), 31624 to upload (86.4 GB), 0 on server with a DIFFERENT size
```

To also check the files that are there by name and size, run the resume command with `--verify` and without
`--dry-run`. If everything is already uploaded it sends nothing, verifies, writes the registry entry if it is missing,
and syncs the shared registry.

From Python, `verify_uploaded_dataset(source_folder, remote_folder_url, auth)` gives the same comparison and returns
the missing, extra and wrong-size files as sets.

## What the tool creates

On the server, under `Robotics-Research-Lab/archival_data/`:

```
LABEL_YYYYMMDD_HASH/
    ...your files and subfolders, hidden files (names starting with a dot) excluded...
    metadata.json     title, source, dates, description, responsible person, hash, archive date
    manifest.csv      one row per file with relative path, size and modification time
    LICENSE.txt       restricted academic-use licence
    README.txt        human-readable description and contact
_registry/upload_registry.csv   shared list of every archived dataset
```

The four sidecar files are uploaded only after every data file is in place, so their presence means the data upload
finished.

In the folder you run the command from:

- `upload_registry.csv`: your local registry. One row is appended when an upload completes (never twice for the same folder).
- `upload_registry_merged.csv`: the merged local and shared registry, as last uploaded to `_registry/`.
- `upload_failures_<folder>_<time>.tsv`: written only when some files failed or differ in size; lists each file and the reason.

What the tool never does: it never deletes or moves anything, locally or on the server, and it never replaces a file
that is already on the server unless you pass `--overwrite-mismatched`. The one file it deliberately replaces is the
shared registry, which it rewrites with the merged version (keeping every row that was on the server). Test folders
and old uploads have to be removed by hand in the u-drive web interface.

## Troubleshooting

**Exit status.** `echo $?` after a run gives 0 when the upload is complete (or after a dry run), 1 when it is incomplete
(run the same command again to resume), and 2 for a fatal error that retrying will not fix, such as a wrong password,
a wrong pin or a missing source folder. With `--strict`, a verification that finds problems ends with an
`AssertionError` instead, so use `--strict` only when you expect the upload to be complete.

**"Authentication failed (HTTP 401)".** The password was not accepted. Retype it; a stray space from pasting is a common
cause. If the web login works but the tool's does not, use an app password (see above). The tool stops at the first
401 on purpose: repeated failed logins can lock the account.

**It seems stuck after the password.** It prints progress every 5 seconds while it scans the local folder and while it
lists the remote folder. If nothing appears for minutes, the dataset may be on a slow or cloud-synced drive whose files
are not all downloaded locally. It never re-hashes when `--version-hash` is given.

**"N consecutive failures after retries; stopping".** The server refused several files in a row even after about two
minutes of retries each, which usually means rate limiting or a network outage. Wait a while and run the same command
again. The option `--max-consecutive-failures` changes the threshold (default 5).

**"Remote folder not found" when resuming.** The label, `--archive-date` or `--version-hash` does not match an existing
folder. Compare them with the folder name on u-drive. Nothing was created.

**Verification lists files "only on server".** Usually files whose names contain `#`, `?` or `%` that were uploaded by the
old version of this tool (before September 2026), which did not encode those characters and stored them under a wrong
name. The current version uploads them correctly; the wrongly named copies have to be removed by hand if wanted.

**Very large single files.** Each network read or write may take up to 300 seconds before it counts as a timeout. If a
very large file keeps failing with timeouts, raise it with `--read-timeout 900`.

## Known issues

`hash_folder` includes hidden files (such as `.DS_Store`) in the hash, while the upload skips them. A hidden file that
appears or changes therefore changes the hash, and with it the folder name, without changing what is uploaded. This is
why a resumed upload should always pin `--version-hash`.

It is left unfixed deliberately, because changing it would change the folder name computed for data that is already
partly uploaded. It is safe to fix once no upload is in progress: every dataset archived so far has completed, verified
and appears in the shared registry. The fix changes the hash only for future archives; the registry keeps the hashes
already recorded, which will not be reproducible with the new function, so record which hash method was used from then on.

`generate_manifest` also lists hidden files, so `manifest.csv` can mention a `.DS_Store` that was not uploaded. This does
not affect folder names and can be fixed at any time.

## Project structure

This repository uses [nbdev](https://nbdev.fast.ai): the code is written in the notebooks under `nbs/` and exported to
the Python package `dataset_archiver/`.

| Notebook | Exports to | Contents |
|---|---|---|
| `nbs/00_core.ipynb` | `dataset_archiver/core.py` | all upload, verification and registry functions |
| `nbs/01_cli.ipynb` | `dataset_archiver/cli.py` | the `upload-dataset` command and its options |
| `nbs/02_resumable.ipynb` | `dataset_archiver/resumable.py` | compatibility module, plus notebook cells for testing against the real server |
| `nbs/index.ipynb` | `README.md` | this page |

Other folders: `data/test_dataset/` is a three-file dataset for trying things out; `nbs/_archive/` holds old copies of the
notebooks (nbdev ignores folders whose names start with `_`); `Claude outputs/` holds offline tests that run against a
simulated WebDAV server.

## Editing the code

The files in `dataset_archiver/` are generated. Edit the notebooks, then export:

```bash
pip install -e .        # once, in the project's Python environment
nbdev_export            # after every change to a notebook
```

After exporting, check that each module was generated from the intended notebook:

```bash
grep -h "File to edit" dataset_archiver/*.py
```

Each line must name `../nbs/00_core.ipynb`, `../nbs/01_cli.ipynb` or `../nbs/02_resumable.ipynb`. Every notebook under
`nbs/` that starts with `#| default_exp core` (or `cli`, `resumable`) is exported to the same module, and the last one
wins. A copy kept for reference under `nbs/` will silently replace the real module unless it sits in a folder whose name
starts with `_`, such as `nbs/_archive/`.

To try changes safely against the real server, use the small test dataset with a test label, a scratch local registry and
no registry sync:

```bash
upload-dataset --source "$PWD/data" --label TEST_something ... \
  --registry /tmp/test_registry.csv --no-sync-registry --verify
```

The offline tests in `Claude outputs/` need no server; see the top of each file for how to run it.

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/Robotics-Research-Lab/dataset-archiver.git
```

or from [conda][conda]

```sh
$ conda install -c Robotics-Research-Lab dataset_archiver
```

or from [pypi][pypi]


```sh
$ pip install dataset_archiver
```


[repo]: https://github.com/Robotics-Research-Lab/dataset-archiver
[docs]: https://Robotics-Research-Lab.github.io/dataset-archiver/
[pypi]: https://pypi.org/project/dataset-archiver/
[conda]: https://anaconda.org/Robotics-Research-Lab/dataset-archiver

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/Robotics-Research-Lab/dataset-archiver
[docs]: https://Robotics-Research-Lab.github.io/dataset-archiver/
[pypi]: https://pypi.org/project/dataset-archiver/
[conda]: https://anaconda.org/Robotics-Research-Lab/dataset-archiver